# Prompting Large Language Models

In this notebook we explore how to interact with a Large Language Model (LLM) through **prompt engineering**. Effective prompting can often replace fine-tuning for many tasks — saving significant compute.

We cover:
1. Loading a quantised LLM efficiently (4-bit / QLoRA)
2. Chat templates
3. Zero-shot prompting
4. Few-shot prompting
5. System prompts
6. Chain-of-Thought (CoT) prompting
7. ReAct prompting

## Setup

In [ ]:
!pip install -q -U trl transformers accelerate
!pip install -q -U datasets bitsandbytes

In [ ]:
# Authenticate to access gated models (e.g. Llama-3)
# You need a HuggingFace account and to accept the model licence at huggingface.co/meta-llama
!huggingface-cli login

## Loading the Model

### Why Llama-3?

We use **Meta Llama-3-8B-Instruct** throughout this notebook. Compared to Llama-2, Llama-3 offers:

- A much larger vocabulary (128k tokens vs 32k) — better multilingual and code coverage
- Improved instruction following out-of-the-box
- A well-defined chat template (using special `<|begin_of_text|>`, `<|eot_id|>` tokens)
- Better performance on standard benchmarks at the same parameter count

With 4-bit quantisation you need roughly **6 GB of VRAM** (vs ~16 GB at bfloat16).

## Quantisation: How to Fit a Large Model in Memory

Neural network weights are floating-point numbers. By default, modern models use **bfloat16** (2 bytes per parameter). An 8B parameter model therefore needs **~16 GB** just to store weights — before activations and optimizer states.

**Quantisation** converts weights to a lower bit-width:

| Precision | Bytes/param | Memory for 8B model |
|---|---|---|
| float32 | 4 | ~32 GB |
| bfloat16 | 2 | ~16 GB |
| int8 | 1 | ~8 GB |
| nf4 (4-bit) | 0.5 | ~4 GB |

### NF4 (NormalFloat 4-bit)
Introduced in the **QLoRA paper** (Dettmers et al., 2023). The 16 quantisation levels are spaced to match the cumulative distribution of normally-distributed weights, minimising quantisation error.

### Double Quantisation
The quantisation constants themselves are also quantised — saving an additional ~0.37 bits per parameter.

### Compute dtype
Weights are stored in 4-bit but computations are performed in `bfloat16` (upcast on the fly), maintaining numerical stability.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # Store weights in 4-bit
    bnb_4bit_quant_type="nf4",       # NormalFloat 4-bit — optimal for normally-distributed weights
    bnb_4bit_use_double_quant=True,  # Quantise the quantisation constants too (~0.37 bits saved/param)
    bnb_4bit_compute_dtype=torch.bfloat16,  # Upcast to bfloat16 for matrix multiplications
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",        # Distribute layers across available GPUs automatically
    trust_remote_code=True,
)
model.config.use_cache = False

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Llama-3 has no dedicated pad token — use eos_token as pad
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

## Chat Templates

Modern instruction-tuned models expect input in a specific structured format called a **chat template**. Using the wrong format leads to degraded performance because the model was fine-tuned to expect a particular delimiter pattern.

HuggingFace tokenizers expose `apply_chat_template()` to handle this automatically. Below we wrap it in a small `chat()` helper used throughout the rest of this notebook.

In [5]:
from transformers import pipeline

# Create the pipeline without generation parameters — the pipeline manages
# its own GenerationConfig internally; passing one at construction causes a conflict.
generator = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
)

def chat(user_message, system_message="You are a helpful assistant.",
         temperature=0.1, max_new_tokens=300, repetition_penalty=1.1):
    """Format a message using the model's chat template and return the assistant reply."""
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message},
    ]
    response = generator(
        messages,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        repetition_penalty=repetition_penalty,
        do_sample=True,
    )
    return response[0]["generated_text"][-1]["content"]

## Prompting Strategies

The sections below cover the main prompting techniques from simplest to most sophisticated.

### 1. Zero-Shot Prompting

Ask the model to perform a task with **no examples** — relying entirely on pre-trained knowledge.

**When to use:** The task is common and well-represented in the training data (translation, summarisation, simple classification).

**Limitations:** Can fail on domain-specific or unusual tasks.

In [6]:
response = chat(
    user_message="Classify the sentiment of this text as positive, neutral, or negative:\n"
                 "'I think the vacation was okay.'"
)
print(response)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I would classify the sentiment of this text as NEUTRAL. The word "okay" is a neutral term that doesn't convey strong emotions or opinions, indicating a lukewarm or average assessment of the vacation.


### 2. Few-Shot Prompting

Provide a handful of labelled examples (**shots**) in the prompt before the actual query. The model performs **in-context learning** — it infers the task format and label space from the examples without any weight updates.

**Key insight:** The examples do not fine-tune the model; they influence only the attention pattern for that single forward pass.

**When to use:** The task is novel, or the zero-shot output format is inconsistent.

In [7]:
few_shot_prompt = """Classify each review as positive, neutral, or negative.

Review: 'This restaurant is the best I've ever been to. Delicious food, friendly staff.'
Sentiment: positive

Review: 'I was disappointed — the product broke within a week.'
Sentiment: negative

Review: 'The movie was okay, not great but not terrible either.'
Sentiment: neutral

Review: 'I absolutely love the new Spider-Man film. Incredibly well done!'
Sentiment:"""

response = chat(user_message=few_shot_prompt)
print(response)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


positive


### 3. System Prompts

The `system` role in a chat template lets you define the model's **persona, rules, and constraints** that persist for the whole conversation. Unlike user instructions, system prompts are typically hidden from end users.

**Use cases:** Chatbots with specific personalities, content-filtered assistants, domain-constrained tools.

In [9]:
system = (
    "You are a grumpy pirate who answers every question correctly "
    "but complains about having to do so. Always end with 'Arrr!'"
)

response = chat(
    user_message="What is the capital of France?",
    system_message=system
)
print(response)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ugh, fine. The capital of France is Paris. Now, can I please get back to me grog and me nap? I've got better things to do than answer questions all day. Arrr!


### 4. Chain-of-Thought (CoT) Prompting

For **reasoning-intensive tasks** (arithmetic, logic, multi-step problems), LLMs perform significantly better when instructed to *reason step by step* before giving the final answer (Wei et al., 2022).

The key insight: generating intermediate reasoning steps provides additional context tokens that guide the model toward the correct answer — essentially using the output space as a scratchpad.

**Variants:**
- **Zero-shot CoT:** Append *"Let's think step by step."* to the prompt.
- **Few-shot CoT:** Include worked examples that show reasoning steps.
- **Self-consistency:** Sample multiple CoT paths and take a majority vote.

In [10]:
problem = (
    "A store sells apples for €1.20 each. A customer buys 3 apples and pays with a €5 note. "
    "How much change does the customer receive?"
)

print("=== Without CoT ===")
print(chat(user_message=problem))

print("\n=== With Zero-Shot CoT ===")
print(chat(user_message=problem + "\n\nLet's think step by step."))

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Without CoT ===


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Let's calculate the total cost of the apples:

3 apples x €1.20 per apple = €3.60

The customer paid with a €5 note, so to find the change, we subtract the cost from the payment:

€5 (payment) - €3.60 (cost) = €1.40

So, the customer receives €1.40 in change.

=== With Zero-Shot CoT ===
I'd be happy to help you calculate the change.

Step 1: Calculate the total cost of the apples
The customer buys 3 apples, and each apple costs €1.20. To find the total cost, multiply the number of apples by the price per apple:

3 apples × €1.20/apple = €3.60

Step 2: Subtract the cost from the €5 note
The customer pays with a €5 note, so subtract the cost of the apples from the €5 note:

€5 - €3.60 = €1.40

So, the customer receives €1.40 in change.


### 5. ReAct Prompting (Reason + Act)

**ReAct** (Yao et al., 2022) interleaves reasoning traces with actions (e.g., tool calls). The model alternates between:
- **Thought:** Internal reasoning about what to do next
- **Action:** An operation to perform (web search, code execution, calculator, …)
- **Observation:** The result of the action

This pattern is the foundation of modern **AI agents** — the model decides which tool to call based on its reasoning, uses the result, and keeps going until the task is complete.

The example below demonstrates the pattern using simulated tool calls:

In [11]:
react_prompt = """Answer the following question using the Thought/Action/Observation pattern.
Available actions: Search[query], Calculate[expression], Finish[answer]

Question: What is the population of the capital of France, and what is it divided by 1000?

Thought: I need to find the capital of France and then look up its population.
Action: Search[capital of France]
Observation: Paris is the capital of France.
Thought: Now I need the population of Paris.
Action: Search[population of Paris]
Observation: Paris has a population of approximately 2,161,000.
Thought: Now I divide 2,161,000 by 1000.
Action: Calculate[2161000 / 1000]
Observation: 2161.0
Thought: I have the final answer.
Action: Finish[2161]"""

print(chat(
    user_message=react_prompt,
    system_message="You are a reasoning assistant that follows the ReAct pattern exactly."
))

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Here's the answer:

What is the population of the capital of France, and what is it divided by 1000?

The capital of France is Paris, which has a population of approximately 2,161,000. Dividing this number by 1000 gives us 2161.0.

Final Answer: The final answer is 2161.0.


### Summary: When to Use Which Prompting Strategy

| Strategy | Best for | Limitations |
|---|---|---|
| Zero-shot | Common, well-defined tasks | May fail on unusual tasks |
| Few-shot | Novel tasks needing format guidance | Uses context window space |
| System prompt | Persona / constraint setting | Model may not always comply |
| Chain-of-Thought | Multi-step reasoning, math | Longer outputs; slower |
| ReAct | Agentic tasks with tools | Requires tool infrastructure |

For more techniques: https://www.promptingguide.ai/techniques

## References

* Quantisation — https://huggingface.co/blog/4bit-transformers-bitsandbytes
* QLoRA paper — https://arxiv.org/pdf/2305.14314.pdf
* Chain-of-Thought prompting (Wei et al., 2022) — https://arxiv.org/abs/2201.11903
* ReAct (Yao et al., 2022) — https://arxiv.org/abs/2210.03629
* Prompting guide — https://www.promptingguide.ai/